# Constitutional Compliance and Economic Growth

**Executable report notebook.** This notebook reads the regenerated pipeline outputs and summarizes the source data, cross-sectional models, panel models, and final interpretation. The Docker workflow exports it to `outputs/report/final_presentation_report.html`.

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import HTML, Image, Markdown, display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

OUTPUTS = ROOT / "outputs"
CROSS = OUTPUTS / "cross_section"
PANEL = OUTPUTS / "panel"
DOCS = ROOT / "docs"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


def load_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Expected generated artifact is missing: {path}")
    return pd.read_csv(path)


def show_table(df: pd.DataFrame, rows: int = 12) -> HTML:
    view = df.head(rows).copy()
    numeric = view.select_dtypes(include="number").columns
    view[numeric] = view[numeric].round(4)
    return HTML(view.to_html(index=False, border=0, classes="dataframe compact"))


def show_figure(path: Path, caption: str) -> None:
    if path.exists():
        display(Markdown(f"**{caption}**"))
        display(Image(filename=str(path), alt=caption))
    else:
        display(Markdown(f"Missing figure: `{path.relative_to(ROOT)}`"))


## 1. What the container did

The Docker default command is `python scripts/generate_report.py`. It validates source files, runs the complete Python reproduction, checks the generated-output manifest, renders the Quarto report/decks, and exports this notebook as HTML.

In [ ]:
summary_path = OUTPUTS / "logs" / "run_all_summary.txt"
text = summary_path.read_text(encoding="utf-8")
status_lines = [line for line in text.splitlines() if line.startswith("## scripts/")]
display(Markdown("\n".join(f"- {line.replace('## ', '')}" for line in status_lines)))


## 2. Source data inventory

The reproduction uses the tracked `.rds` and `.dta` inputs from the original R project. The inventory below proves that the pipeline inspected the source data before model regeneration.

In [ ]:
inventory = load_csv(DOCS / "data_inventory.csv")
cols = ["group", "dataset", "relative_path", "rows", "columns", "missing_cells"]
show_table(inventory[cols], rows=15)


## 3. Cross-sectional workflow

The cross-sectional part reproduces the 2010-2019 OLS workflow. The preferred sample is the outlier-filtered design used for the main robustness table.

In [ ]:
show_table(load_csv(CROSS / "cross_section_preparation_summary.csv"))


In [ ]:
ols = load_csv(CROSS / "ols_main.csv")
main_terms = ["log_cc_total", "log_GDPpc2015", "investment", "gov_exp_reduced"]
main_ols = ols[(ols["model"] == "re4_o") & (ols["term"].isin(main_terms))]
main_ols = main_ols[["term", "coefficient", "robust_se_hc3", "p_value", "stars", "nobs", "r_squared"]]
show_table(main_ols, rows=10)


**Cross-sectional conclusion.** The coefficient on `log(cc_total + 2)` is small and statistically insignificant in the preferred OLS specification. This reproduces the original paper's cross-sectional result.

In [ ]:
institutional = load_csv(CROSS / "ols_institutional_variants.csv")
institution_terms = ["rule_of_law", "ctrl_of_corr", "gov_eff", "pol_stab", "reg_q", "voice_and_acc"]
institutional = institutional[institutional["term"].isin(institution_terms)]
show_table(institutional[["model", "term", "coefficient", "robust_se_hc3", "p_value", "stars"]], rows=10)


The institutional alternatives are important for interpretation: broader governance indicators often correlate with growth, but the constitutional-compliance index does not in the cross-section.

In [ ]:
show_figure(CROSS / "figures" / "correlation_plot.png", "Cross-sectional Spearman correlation plot")


## 4. Panel workflow

The panel part reproduces the 1990-2020 country-year workflow. The specification tests reject pooled OLS and random effects, so the headline interpretation comes from fixed effects with robust covariance.

In [ ]:
show_table(load_csv(PANEL / "specification_tests.csv"), rows=10)


In [ ]:
show_table(load_csv(PANEL / "diagnostic_tests.csv"), rows=10)


Serial correlation, heteroskedasticity, and cross-sectional dependence are present, so the report emphasizes Driscoll-Kraay standard errors.

In [ ]:
fe = load_csv(PANEL / "fixed_effects_main.csv")
headline_terms = ["log_cc_total", "fertility", "lag_log_GDPpc2015", "gov_exp_reduced", "investment"]
fe = fe[fe["term"].isin(headline_terms)]
show_table(fe[["term", "coefficient", "se_driscoll_kraay", "se_classical"]], rows=10)


**Panel conclusion.** The coefficient on `log_cc_total` is positive and statistically meaningful in the fixed-effects panel model. The main result is approximately `+1.74` percentage points with Driscoll-Kraay SE around `0.29`, matching the original paper in sign, magnitude, and conclusion.

In [ ]:
components = load_csv(PANEL / "fixed_effects_compliance_categories.csv")
component_terms = ["log_cc_basic", "log_cc_civil", "log_cc_polit", "log_cc_prop"]
components = components[components["term"].isin(component_terms)]
show_table(components[["compliance_variant", "term", "coefficient", "se_driscoll_kraay", "p_value", "nobs"]], rows=10)


In [ ]:
show_figure(PANEL / "figures" / "correlation_plot.png", "Panel Spearman correlation plot")


## 5. Final interpretation

- The **cross-section** asks whether high-compliance countries grow faster than low-compliance countries after controls. The answer is no.
- The **panel** asks whether a country grows faster after its own compliance improves. The answer is yes.
- The Docker image makes this result reproducible from a clean, public runtime.